In [15]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASS = os.getenv("DB_PASS")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASS}@{DB_HOST}:5432/{DB_NAME}"
)

In [3]:
df = pd.read_sql("SELECT * FROM customer_behavior_cleaned", engine)

print("Shape:", df.shape)
df.head()


Shape: (1000000, 63)


,user_id,age,gender,country,urban_rural,income_level,employment_status,education_level,relationship_status,has_children,...,app_usage_frequency,notification_response_rate,account_age_months,last_purchase_date,social_sharing_frequency,premium_subscription,return_rate,age_group,income_band,spend_level
0,1,56,female,germany,suburban,90860,self-employed,associate degree,single,0,...,7,74,19,2025-06-22,6,1,50,51+,medium,Medium
1,2,69,male,japan,suburban,35423,unemployed,bachelor,single,1,...,5,23,8,2026-07-25,3,0,37,51+,low,Medium
2,3,46,female,india,urban,21467,self-employed,associate degree,married,1,...,7,12,13,2026-02-26,6,0,53,36-50,low,Medium
3,4,32,male,canada,urban,41770,self-employed,bachelor,widowed,0,...,4,19,9,2026-10-27,7,0,98,26-35,low,Low
4,5,60,female,japan,urban,183882,employed,associate degree,widowed,1,...,7,30,3,2026-06-23,3,0,86,51+,high,Medium


In [4]:
features = [
    'monthly_spend',
    'average_order_value',
    'cart_abandonment_rate',
    'browse_to_buy_ratio',
    'impulse_buying_score'
]

df_model = df[features]

In [13]:

import numpy as np

impulse_median = df["impulse_buying_score"].median()

conditions = [
    df["impulse_buying_score"] > impulse_median * 1.2,
    df["impulse_buying_score"] > impulse_median,
    df["impulse_buying_score"] <= impulse_median
]

choices = [
    "loyal_customers",
    "potential_loyalists",
    "at_risk_customers"
]

df["customer_segment"] = np.select(conditions, choices,default="general_customers")

In [6]:
df["customer_segment"].value_counts()

customer_segment
at_risk_customers      545613
loyal_customers        363878
potential_loyalists     90509
Name: count, dtype: int64

In [7]:
df["customer_segment"].value_counts(normalize=True) * 100

customer_segment
at_risk_customers      54.5613
loyal_customers        36.3878
potential_loyalists     9.0509
Name: proportion, dtype: float64

In [8]:

recommendation_map = {
    "loyal_customers": "Premium product recommendations",
    "potential_loyalists": "Personalized bundle offers",
    "at_risk_customers": "Discount and retention offers",
}

df["recommendation_strategy"] = df["customer_segment"].map(recommendation_map)

In [9]:
df[["customer_segment", "recommendation_strategy"]].head()

,customer_segment,recommendation_strategy
0,at_risk_customers,Discount and retention offers
1,at_risk_customers,Discount and retention offers
2,at_risk_customers,Discount and retention offers
3,at_risk_customers,Discount and retention offers
4,loyal_customers,Premium product recommendations


In [ ]:
df.to_sql(
    "customer_behavior_final",
    engine,
    if_exists="replace",
    index=False
)